# Named Entity Recognition (NER) on Social Media Tweets
### NLP Assignment — Rule-Based Approach using spaCy

---

## 1. Introduction

**Named Entity Recognition (NER)** is a fundamental NLP task that identifies and classifies named entities in text into predefined categories such as:

| Entity Type | Description | Example |
|-------------|-------------|----------|
| `PERSON` | People's names | Elon Musk |
| `ORG` | Organizations, companies | Tesla, NASA |
| `GPE` | Countries, cities, states | Nepal, London |
| `LOC` | Non-GPE locations | Mount Everest |
| `DATE` | Dates and periods | March 2024 |
| `PRODUCT` | Products, objects | iPhone 15 |
| `EVENT` | Named events | World Cup |

**Objective:** Apply spaCy's pre-trained rule-based NER model on a dataset of tweets to extract and analyze named entities.

**Tools Used:**
- `spaCy` — NLP library with pre-trained English model
- `pandas` — Data manipulation
- `matplotlib` / `collections` — Visualization

---
## 2. Setup & Installation

In [1]:
# Install required libraries (run once)
!pip install spacy pandas matplotlib
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "c:\Users\aarav\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\aarav\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\aarav\AppData\Local\Programs\Python\Python310\lib\site-packages\spacy\__main__.py", line 4, in <module>
    setup_cli()
  File "c:\Users\aarav\AppData\Local\Programs\Python\Python310\lib\site-packages\spacy\cli\_util.py", line 84, in setup_cli
    registry.cli.get_all()
  File "c:\Users\aarav\AppData\Local\Programs\Python\Python310\lib\site-packages\catalogue\__init__.py", line 110, in get_all
    result.update(self.get_entry_points())
  File "c:\Users\aarav\AppData\Local\Programs\Python\Python310\lib\site-packages\catalogue\__init

In [3]:
# Import libraries
import spacy
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re

# Load spaCy English model
nlp = spacy.load('en_core_web_sm')

print('spaCy version:', spacy.__version__)
print('Model loaded: en_core_web_sm')

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

---
## 3. Dataset

We use a curated set of realistic sample tweets covering technology, sports, politics, and entertainment.

In [ ]:
# Sample tweet dataset
tweets = [
    "Elon Musk just announced that Tesla will open a new Gigafactory in India by 2026.",
    "The United Nations held an emergency meeting in New York to discuss climate change.",
    "Lionel Messi scored a hat-trick for Inter Miami in the MLS Cup final last Saturday.",
    "Apple unveiled the iPhone 16 at its Worldwide Developers Conference in San Francisco.",
    "Prime Minister Narendra Modi visited Nepal to strengthen ties between India and Nepal.",
    "NASA's Artemis mission is planning to land astronauts on the Moon in December 2025.",
    "Jeff Bezos and Amazon are investing $1 billion in renewable energy projects in Africa.",
    "The Paris Olympics 2024 saw Team USA dominate the swimming events at Seine Saint-Denis.",
    "OpenAI and Microsoft are collaborating to bring GPT-5 to Azure cloud services.",
    "Cristiano Ronaldo broke the all-time goal record playing for Al Nassr in Saudi Arabia.",
    "The World Health Organization warned about a new flu variant detected in Southeast Asia.",
    "Taylor Swift's Eras Tour became the highest-grossing concert tour in history last year.",
    "Google DeepMind published a groundbreaking AI research paper in Nature this week.",
    "The earthquake in Turkey and Syria left thousands homeless in February last year.",
    "Sundar Pichai confirmed that Google will lay off 12,000 employees across North America.",
    "SpaceX successfully launched the Starship rocket from Boca Chica, Texas, on Tuesday.",
    "The FIFA World Cup 2026 will be co-hosted by the United States, Canada, and Mexico.",
    "Mark Zuckerberg announced Meta's new AR glasses at a conference in Los Angeles.",
    "The Reserve Bank of India raised interest rates in Mumbai to combat rising inflation.",
    "Volodymyr Zelensky addressed the European Parliament in Brussels about the Ukraine war."
]

# Create a DataFrame
df = pd.DataFrame({'tweet_id': range(1, len(tweets)+1), 'text': tweets})
print(f'Dataset loaded: {len(df)} tweets')
df.head()

---
## 4. Text Preprocessing

Tweets often contain noise like hashtags, mentions, and URLs. We clean the text before applying NER.

In [ ]:
def preprocess_tweet(text):
    """Clean tweet text for NER processing."""
    text = re.sub(r'http\S+|www\S+', '', text)       # Remove URLs
    text = re.sub(r'@\w+', '', text)                  # Remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)             # Remove # but keep word
    text = re.sub(r'\s+', ' ', text).strip()          # Normalize whitespace
    return text

df['clean_text'] = df['text'].apply(preprocess_tweet)

# Show before/after example
sample = "Check out @elonmusk's latest #Tesla announcement! https://t.co/abc123"
print('Before:', sample)
print('After: ', preprocess_tweet(sample))

---
## 5. Applying NER with spaCy

spaCy's `en_core_web_sm` model uses a rule-based pipeline combined with statistical components trained on news/web text. It identifies entity spans and assigns labels automatically.

In [ ]:
def extract_entities(text):
    """Extract named entities from text using spaCy."""
    doc = nlp(text)
    entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]
    return entities

# Apply to all tweets
df['entities'] = df['clean_text'].apply(extract_entities)

# Display results for first 5 tweets
for _, row in df.head(5).iterrows():
    print(f"\nTweet {row['tweet_id']}: {row['text']}")
    if row['entities']:
        for ent_text, label, description in row['entities']:
            print(f"   [{label}] '{ent_text}' — {description}")
    else:
        print("   No entities found")

---
## 6. Visualization — displaCy Entity Renderer

In [ ]:
from spacy import displacy

# Visualize entity spans on a sample tweet
sample_tweet = df['clean_text'].iloc[7]  # Paris Olympics tweet
doc = nlp(sample_tweet)

print('Tweet:', sample_tweet)
displacy.render(doc, style='ent', jupyter=True)

---
## 7. Entity Frequency Analysis

In [ ]:
# Flatten all entities across all tweets
all_entities = []
for ents in df['entities']:
    for ent_text, label, _ in ents:
        all_entities.append({'entity': ent_text, 'label': label})

ent_df = pd.DataFrame(all_entities)

print(f'Total entities found: {len(ent_df)}')
print(f'Unique entities: {ent_df["entity"].nunique()}')
print(f'\nEntity type distribution:')
print(ent_df['label'].value_counts())

In [ ]:
# Plot entity type distribution and top entities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NER Analysis on Tweets — spaCy (en_core_web_sm)', fontsize=14, fontweight='bold')

# Chart 1: Entity type counts
label_counts = ent_df['label'].value_counts()
colors = plt.cm.Set3.colors[:len(label_counts)]
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Entity Type Distribution')
axes[0].set_xlabel('Entity Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Chart 2: Top 10 most frequent entities
top_entities = ent_df['entity'].value_counts().head(10)
axes[1].barh(top_entities.index[::-1], top_entities.values[::-1], color='steelblue', edgecolor='black')
axes[1].set_title('Top 10 Most Frequent Entities')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('ner_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as ner_analysis.png')

---
## 8. Per-Category Entity Inspection

In [ ]:
# Show top entities per category
categories = ['PERSON', 'ORG', 'GPE', 'DATE', 'PRODUCT', 'EVENT', 'LOC']

for cat in categories:
    subset = ent_df[ent_df['label'] == cat]['entity'].value_counts().head(5)
    if not subset.empty:
        print(f"\n{cat} — {spacy.explain(cat)}")
        for name, count in subset.items():
            print(f"   {name}: {count}")

---
## 9. Structured Results Table

In [ ]:
# Build a structured results table
rows = []
for _, row in df.iterrows():
    for ent_text, label, description in row['entities']:
        rows.append({
            'tweet_id': row['tweet_id'],
            'tweet_snippet': row['text'][:60] + '...',
            'entity': ent_text,
            'label': label,
            'description': description
        })

results_df = pd.DataFrame(rows)
print(f'Total entity records: {len(results_df)}')
results_df.head(15)

---
## 10. Evaluation & Discussion

### 10.1 Observations
- spaCy's `en_core_web_sm` correctly identified most **PERSON**, **ORG**, and **GPE** entities in the tweets.
- Entities like *Elon Musk*, *NASA*, *Paris*, and *Nepal* were accurately tagged.
- Some newer entities (e.g., *OpenAI*, *Meta*) may be missed or mislabeled since the model was trained predominantly on news text, not social media.

### 10.2 Limitations of Rule-Based / Pre-trained NER on Tweets

| Limitation | Description |
|------------|-------------|
| Informal language | Tweets use slang, abbreviations, and hashtags |
| Ambiguity | "Apple" could be the fruit or the company |
| Emerging entities | New brands/people not in training data |
| Short context | Tweets are 280 chars — limited context for disambiguation |

### 10.3 Possible Improvements
- Fine-tune a BERT-based NER model on Twitter-specific datasets (e.g., **WNUT-17**)
- Use `en_core_web_lg` (large model) for better accuracy
- Add custom entity rules using spaCy's `EntityRuler`

---
## 11. Conclusion

In this assignment, we successfully applied **Named Entity Recognition** on 20 tweets using **spaCy's pre-trained model**. We:

1. Preprocessed raw tweet text (removing URLs, mentions, hashtags)
2. Applied spaCy's `en_core_web_sm` NER pipeline
3. Extracted and categorized entities (PERSON, ORG, GPE, DATE, etc.)
4. Visualized entity distributions and top recurring entities
5. Discussed limitations and potential improvements

NER on social media text remains a challenging but highly valuable task with real-world applications in **brand monitoring**, **event detection**, **news aggregation**, and **social analytics**.

In [ ]:
# Export results to CSV
results_df.to_csv('ner_results.csv', index=False)
print('Results exported to ner_results.csv')